In [2]:
import numpy as np
import openrouteservice
import folium
import networkx


In [2]:
apiKey = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImU2NzNmZmZiMmFkMzRjNGNhZDlhZjFlZWNhN2JmZmZmIiwiaCI6Im11cm11cjY0In0="
client = openrouteservice.Client(key=apiKey)


In [3]:
# tu sie napierdala punkty paczkomatow itd.
points = [
    (15.495625, 51.940579),  # Castorama (baza)

    (15.492387, 51.942214), #wiejska
    (15.495385,51.937817)
]

In [4]:
route = client.directions(
    coordinates=points,
    profile='driving-car',
    format='geojson'
)

In [5]:
summary = route['features'][0]['properties']['summary']

m = folium.Map(location=[51.9356, 15.5064], zoom_start=13)

folium.GeoJson(route).add_to(m)

for lon, lat in points:
    folium.Marker([lat, lon]).add_to(m)

m.save("mapa_trasy.html")

In [ ]:
import math
import pandas as pd
import folium
import networkx as nx


def calculateDistanceKm(lat1, lon1, lat2, lon2):
    earthRadiusKm = 6371

    lat1Rad = math.radians(lat1)
    lon1Rad = math.radians(lon1)
    lat2Rad = math.radians(lat2)
    lon2Rad = math.radians(lon2)

    latDiff = lat2Rad - lat1Rad
    lonDiff = lon2Rad - lon1Rad

    a = (
        math.sin(latDiff / 2) ** 2
        + math.cos(lat1Rad) * math.cos(lat2Rad) * math.sin(lonDiff / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return earthRadiusKm * c


def getMarkerColor(pointType):
    if pointType == "magazyn":
        return "red"
    if pointType == "paczkomat":
        return "blue"
    if pointType == "klient":
        return "green"
    return "gray"


def createGraph(pointsDf):
    graph = nx.Graph()

    for _, row in pointsDf.iterrows():
        graph.add_node(
            row["id"],
            name=row["nazwa"],
            lat=row["lat"],
            lon=row["lon"],
            type=row["typ"]
        )

    for i in range(len(pointsDf)):
        for j in range(i + 1, len(pointsDf)):
            pointA = pointsDf.iloc[i]
            pointB = pointsDf.iloc[j]

            distanceKm = calculateDistanceKm(
                pointA["lat"],
                pointA["lon"],
                pointB["lat"],
                pointB["lon"]
            )

            graph.add_edge(
                pointA["id"],
                pointB["id"],
                distanceKm=round(distanceKm, 3),
                weight=round(distanceKm, 3)
            )

    return graph


def createMap(pointsDf, graph, route=None):
    centerLat = pointsDf["lat"].mean()
    centerLon = pointsDf["lon"].mean()

    courierMap = folium.Map(
        location=[centerLat, centerLon],
        zoom_start=13
    )

    for _, row in pointsDf.iterrows():
        popupText = f"""
        <b>{row["nazwa"]}</b><br>
        ID: {row["id"]}<br>
        Typ: {row["typ"]}<br>
        Lat: {row["lat"]}<br>
        Lon: {row["lon"]}
        """

        folium.Marker(
            location=[row["lat"], row["lon"]],
            popup=popupText,
            tooltip=row["nazwa"],
            icon=folium.Icon(color=getMarkerColor(row["typ"]))
        ).add_to(courierMap)

    if route is not None:
        for i in range(len(route) - 1):
            pointAId = route[i]
            pointBId = route[i + 1]

            pointA = graph.nodes[pointAId]
            pointB = graph.nodes[pointBId]

            distanceKm = graph[pointAId][pointBId]["distanceKm"]

            folium.PolyLine(
                locations=[
                    [pointA["lat"], pointA["lon"]],
                    [pointB["lat"], pointB["lon"]]
                ],
                tooltip=f"{pointAId} → {pointBId}: {distanceKm} km",
                weight=4
            ).add_to(courierMap)

    return courierMap


def calculateRouteDistance(graph, route):
    totalDistance = 0

    for i in range(len(route) - 1):
        pointA = route[i]
        pointB = route[i + 1]
        totalDistance += graph[pointA][pointB]["distanceKm"]

    return round(totalDistance, 3)


def main():
    pointsDf = pd.read_csv("punkty.csv")

    graph = createGraph(pointsDf)

    route = ["M1", "P1", "P3", "P4", "P2", "M1"]

    totalDistance = calculateRouteDistance(graph, route)

    courierMap = createMap(pointsDf, graph, route)
    courierMap.save("mapa_trasy.html")

    print("Mapa zapisana jako: mapa_trasy.html")
    print("Trasa:", " -> ".join(route))
    print("Długość trasy:", totalDistance, "km")


if __name__ == "__main__":
    main()

Mapa zapisana jako: mapa_trasy.html
Trasa: M1 -> P1 -> P3 -> P4 -> P2 -> M1
Długość trasy: 4.717 km


In [1]:
import pandas as pd
import folium
import osmnx as ox
import networkx as nx


def createRoadGraph(pointsDf):
    north = pointsDf["lat"].max() + 0.01
    south = pointsDf["lat"].min() - 0.01
    east = pointsDf["lon"].max() + 0.01
    west = pointsDf["lon"].min() - 0.01

    
    roadGraph = ox.graph_from_place(
        "Zielona Góra, Lubusz Voivodeship, Poland",
        network_type="drive"
)

    return roadGraph


def getNearestRoadNode(roadGraph, lat, lon):
    return ox.distance.nearest_nodes(
        roadGraph,
        X=lon,
        Y=lat
    )


def getRouteCoordinates(roadGraph, routeNodes):
    coordinates = []

    for node in routeNodes:
        lat = roadGraph.nodes[node]["y"]
        lon = roadGraph.nodes[node]["x"]
        coordinates.append([lat, lon])

    return coordinates


def createMap(pointsDf, roadGraph, routeIds):
    centerLat = pointsDf["lat"].mean()
    centerLon = pointsDf["lon"].mean()

    courierMap = folium.Map(
        location=[centerLat, centerLon],
        zoom_start=13
    )

    pointData = {}

    for _, row in pointsDf.iterrows():
        nearestNode = getNearestRoadNode(
            roadGraph,
            row["lat"],
            row["lon"]
        )

        pointData[row["id"]] = {
            "name": row["nazwa"],
            "lat": row["lat"],
            "lon": row["lon"],
            "type": row["typ"],
            "roadNode": nearestNode
        }

        markerColor = "blue"

        if row["typ"] == "magazyn":
            markerColor = "red"

        folium.Marker(
            location=[row["lat"], row["lon"]],
            tooltip=row["nazwa"],
            popup=f"""
            <b>{row["nazwa"]}</b><br>
            ID: {row["id"]}<br>
            Typ: {row["typ"]}
            """,
            icon=folium.Icon(color=markerColor)
        ).add_to(courierMap)

    totalDistanceMeters = 0

    for i in range(len(routeIds) - 1):
        startId = routeIds[i]
        endId = routeIds[i + 1]

        startNode = pointData[startId]["roadNode"]
        endNode = pointData[endId]["roadNode"]

        shortestPath = nx.shortest_path(
            roadGraph,
            startNode,
            endNode,
            weight="length"
        )

        distanceMeters = nx.shortest_path_length(
            roadGraph,
            startNode,
            endNode,
            weight="length"
        )

        totalDistanceMeters += distanceMeters

        routeCoordinates = getRouteCoordinates(roadGraph, shortestPath)

        folium.PolyLine(
            locations=routeCoordinates,
            weight=5,
            tooltip=f"{startId} → {endId}: {distanceMeters / 1000:.2f} km"
        ).add_to(courierMap)

    courierMap.save("mapa_po_drogach.html")

    print("Mapa zapisana jako: mapa_po_drogach.html")
    print("Długość trasy:", round(totalDistanceMeters / 1000, 2), "km")


def main():
    pointsDf = pd.read_csv("punkty.csv")

    roadGraph = createRoadGraph(pointsDf)

    routeIds = ["M1", "P1", "P3", "P4", "P2", "M1"]

    createMap(pointsDf, roadGraph, routeIds)


if __name__ == "__main__":
    main()

Mapa zapisana jako: mapa_po_drogach.html
Długość trasy: 6.92 km


In [2]:
import math
import folium
import networkx as nx


points = [
    {"id": "M1", "name": "Magazyn", "lat": 51.935, "lon": 15.506},
    {"id": "P1", "name": "Paczkomat 1", "lat": 51.942, "lon": 15.520},
    {"id": "P2", "name": "Paczkomat 2", "lat": 51.927, "lon": 15.514},
    {"id": "P3", "name": "Paczkomat 3", "lat": 51.922, "lon": 15.498},
]


def calculateDistanceKm(lat1, lon1, lat2, lon2):
    r = 6371

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return r * c


G = nx.Graph()

for point in points:
    G.add_node(
        point["id"],
        name=point["name"],
        lat=point["lat"],
        lon=point["lon"]
    )


for i in range(len(points)):
    for j in range(i + 1, len(points)):

        p1 = points[i]
        p2 = points[j]

        distanceKm = calculateDistanceKm(
            p1["lat"],
            p1["lon"],
            p2["lat"],
            p2["lon"]
        )

        estimatedTime = distanceKm * 2

        traffic = round(distanceKm / 10, 2)

        difficulty = round(
            0.5 * traffic +
            0.5 * (distanceKm / 5),
            2
        )

        G.add_edge(
            p1["id"],
            p2["id"],
            distanceKm=round(distanceKm, 2),
            estimatedTimeMin=round(estimatedTime, 2),
            trafficLevel=traffic,
            difficulty=difficulty
        )


m = folium.Map(
    location=[51.935, 15.506],
    zoom_start=13
)


for node, data in G.nodes(data=True):

    folium.Marker(
        [data["lat"], data["lon"]],
        tooltip=data["name"]
    ).add_to(m)


for u, v, data in G.edges(data=True):

    node1 = G.nodes[u]
    node2 = G.nodes[v]

    difficulty = data["difficulty"]

    color = "green"

    if difficulty > 0.5:
        color = "orange"

    if difficulty > 1:
        color = "red"

    folium.PolyLine(
        locations=[
            [node1["lat"], node1["lon"]],
            [node2["lat"], node2["lon"]]
        ],
        color=color,
        weight=4,
        tooltip=(
            f"{u} → {v}<br>"
            f"Dystans: {data['distanceKm']} km<br>"
            f"Czas: {data['estimatedTimeMin']} min<br>"
            f"Trudność: {data['difficulty']}"
        )
    ).add_to(m)


m.save("graf_logistyczny.html")

print("Zapisano graf_logistyczny.html")

Zapisano graf_logistyczny.html


In [10]:
import math
import pandas as pd
import folium
import networkx as nx


points = [
    {
        "id": "M1",
        "name": "Zielona Góra",
        "lat": 51.9354800,
        "lon": 15.5064300,
        "type": "baza",
        "parcelLoadKg": 0,
        "stopTimeMin": 0,
        "timeWindowStart": "08:00",
        "timeWindowEnd": "18:00",
        "stopComplexity": 0.1
    },
    {
        "id": "P1",
        "name": "Nowogród Bobrzański",
        "lat": 51.798346,
        "lon": 15.236140,
        "type": "miasto",
        "parcelLoadKg": 42,
        "stopTimeMin": 6,
        "timeWindowStart": "09:00",
        "timeWindowEnd": "12:00",
        "stopComplexity": 0.6
    },
    {
        "id": "P2",
        "name": "Żary",
        "lat": 51.636,
        "lon": 15.138,
        "type": "miasto",
        "parcelLoadKg": 31,
        "stopTimeMin": 5,
        "timeWindowStart": "10:00",
        "timeWindowEnd": "14:00",
        "stopComplexity": 0.5
    },
    {
        "id": "P3",
        "name": "Żagań",
        "lat": 51.616591,
        "lon": 15.317660,
        "type": "miasto",
        "parcelLoadKg": 55,
        "stopTimeMin": 8,
        "timeWindowStart": "08:30",
        "timeWindowEnd": "11:30",
        "stopComplexity": 0.8
    },
    {
        "id": "P4",
        "name": "Szprotawa",
        "lat": 51.567,
        "lon": 15.538,
        "type": "miasto",
        "parcelLoadKg": 27,
        "stopTimeMin": 4,
        "timeWindowStart": "12:00",
        "timeWindowEnd": "16:00",
        "stopComplexity": 0.4
    },
    {
        "id": "P5",
        "name": "Kożuchów",
        "lat": 51.745,
        "lon": 15.595,
        "type": "miasto",
        "parcelLoadKg": 63,
        "stopTimeMin": 9,
        "timeWindowStart": "09:30",
        "timeWindowEnd": "13:30",
        "stopComplexity": 0.9
    },

    {
        "id":"P6",
        "name": "Nowa Sól",
        "lat": 51.8011,
        "lon": 15.7075,
        "type": "miasto",
        "parcelLoadKg": 63,
        "stopTimeMin": 9,
        "timeWindowStart": "09:30",
        "timeWindowEnd": "13:30",
        "stopComplexity": 0.9
    }
]


def calculateDistanceKm(lat1, lon1, lat2, lon2):
    earthRadiusKm = 6371

    lat1Rad = math.radians(lat1)
    lon1Rad = math.radians(lon1)
    lat2Rad = math.radians(lat2)
    lon2Rad = math.radians(lon2)

    latDiff = lat2Rad - lat1Rad
    lonDiff = lon2Rad - lon1Rad

    a = (
        math.sin(latDiff / 2) ** 2
        + math.cos(lat1Rad)
        * math.cos(lat2Rad)
        * math.sin(lonDiff / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return earthRadiusKm * c


def normalize(value, minValue, maxValue):
    if maxValue == minValue:
        return 0

    return (value - minValue) / (maxValue - minValue)


def getEdgeColor(difficulty):
    if difficulty < 0.35:
        return "green"
    if difficulty < 0.65:
        return "orange"
    return "red"


def getMarkerColor(pointType):
    if pointType == "baza":
        return "red"
    if pointType == "miasto":
        return "blue"
    return "gray"


def buildLogisticGraph(pointsList):
    graph = nx.Graph()

    for point in pointsList:
        graph.add_node(
            point["id"],
            name=point["name"],
            lat=point["lat"],
            lon=point["lon"],
            type=point["type"],
            liczbaPaczkomatow =['liczba paczkomatow']
        )

    rawEdges = []

    for i in range(len(pointsList)):
        for j in range(i + 1, len(pointsList)):
            pointA = pointsList[i]
            pointB = pointsList[j]

            distanceKm = calculateDistanceKm(
                pointA["lat"],
                pointA["lon"],
                pointB["lat"],
                pointB["lon"]
            )

            averageCitySpeedKmH = 32
            estimatedTimeMin = (distanceKm / averageCitySpeedKmH) * 60


            rawEdges.append({
                "from": pointA["id"],
                "to": pointB["id"],
                "distanceKm": distanceKm,
                "estimatedTimeMin": estimatedTimeMin,
            })

    minDistance = min(edge["distanceKm"] for edge in rawEdges)
    maxDistance = max(edge["distanceKm"] for edge in rawEdges)

    minTime = min(edge["estimatedTimeMin"] for edge in rawEdges)
    maxTime = max(edge["estimatedTimeMin"] for edge in rawEdges)

    minLoad = min(edge["averageParcelLoadKg"] for edge in rawEdges)
    maxLoad = max(edge["averageParcelLoadKg"] for edge in rawEdges)

    for edge in rawEdges:
        distanceScore = normalize(edge["distanceKm"], minDistance, maxDistance)
        timeScore = normalize(edge["estimatedTimeMin"], minTime, maxTime)
        loadScore = normalize(edge["averageParcelLoadKg"], minLoad, maxLoad)
        stopScore = edge["averageStopComplexity"]

        difficulty = (
            0.35 * distanceScore
            + 0.30 * timeScore
            + 0.20 * stopScore
            + 0.15 * loadScore
        )

        fuelUsageLiters = edge["distanceKm"] * 0.09
        fuelCostPln = fuelUsageLiters * 6.50

        graph.add_edge(
            edge["from"],
            edge["to"],
            distanceKm=round(edge["distanceKm"], 3),
            estimatedTimeMin=round(edge["estimatedTimeMin"], 2),
            averageStopComplexity=round(edge["averageStopComplexity"], 2),
            averageParcelLoadKg=round(edge["averageParcelLoadKg"], 2),
            fuelUsageLiters=round(fuelUsageLiters, 2),
            fuelCostPln=round(fuelCostPln, 2),
            difficulty=round(difficulty, 3),
            weight=round(difficulty, 3)
        )

    return graph


def createMap(graph, outputFileName):
    nodeData = list(graph.nodes(data=True))

    centerLat = sum(data["lat"] for _, data in nodeData) / len(nodeData)
    centerLon = sum(data["lon"] for _, data in nodeData) / len(nodeData)

    logisticMap = folium.Map(
    location=[centerLat, centerLon],
    zoom_start=13,
    tiles="CartoDB positron"
)

    for nodeId, data in graph.nodes(data=True):
        popupText = f"""
        <b>{data["name"]}</b><br>
        ID: {nodeId}<br>
        Typ: {data["type"]}<br>
        Masa paczek: {data["parcelLoadKg"]} kg<br>
        Czas postoju: {data["stopTimeMin"]} min<br>
        Okno czasowe: {data["timeWindowStart"]} - {data["timeWindowEnd"]}<br>
        Złożoność postoju: {data["stopComplexity"]}
        """

        folium.Marker(
            location=[data["lat"], data["lon"]],
            tooltip=f"{nodeId}: {data['name']}",
            popup=folium.Popup(popupText, max_width=300),
            icon=folium.Icon(color=getMarkerColor(data["type"]))
        ).add_to(logisticMap)

    for fromNode, toNode, data in graph.edges(data=True):
        nodeA = graph.nodes[fromNode]
        nodeB = graph.nodes[toNode]

        difficulty = data["difficulty"]
        color = getEdgeColor(difficulty)

        tooltipText = f"""
        {fromNode} → {toNode}<br>
        Dystans: {data["distanceKm"]} km<br>
        Czas szacowany: {data["estimatedTimeMin"]} min<br>
        Paliwo: {data["fuelUsageLiters"]} l<br>
        Koszt paliwa: {data["fuelCostPln"]} zł<br>
        Średnia masa paczek: {data["averageParcelLoadKg"]} kg<br>
        Trudność: {data["difficulty"]}
        """

        folium.PolyLine(
            locations=[
                [nodeA["lat"], nodeA["lon"]],
                [nodeB["lat"], nodeB["lon"]]
            ],
            color=color,
            weight=3 + difficulty * 5,
            opacity=0.75,
            tooltip=tooltipText
        ).add_to(logisticMap)

    legendHtml = """
    <div style="
        position: fixed;
        bottom: 40px;
        left: 40px;
        width: 230px;
        background-color: white;
        border: 2px solid gray;
        z-index: 9999;
        padding: 10px;
        font-size: 14px;
    ">
        <b>Legenda trudności odcinka</b><br>
        <span style="color: green;">■</span> łatwy<br>
        <span style="color: orange;">■</span> średni<br>
        <span style="color: red;">■</span> trudny<br>
        <br>
        Grubsza linia = większa trudność
    </div>
    """

    logisticMap.get_root().html.add_child(folium.Element(legendHtml))
    logisticMap.save(outputFileName)


def exportGraphToCsv(graph):
    nodesRows = []

    for nodeId, data in graph.nodes(data=True):
        row = {"id": nodeId}
        row.update(data)
        nodesRows.append(row)

    edgesRows = []

    for fromNode, toNode, data in graph.edges(data=True):
        row = {
            "from": fromNode,
            "to": toNode
        }
        row.update(data)
        edgesRows.append(row)

    nodesDf = pd.DataFrame(nodesRows)
    edgesDf = pd.DataFrame(edgesRows)

    nodesDf.to_csv("nodes_paczkomaty.csv", index=False)
    edgesDf.to_csv("edges_wagi_trudnosci.csv", index=False)


def main():
    graph = buildLogisticGraph(points)

    createMap(
        graph=graph,
        outputFileName="mapa_paczkomatow_trudnosc.html"
    )

    exportGraphToCsv(graph)

    print("Zapisano pliki:")
    print("- mapa_paczkomatow_trudnosc.html")
    print("- nodes_paczkomaty.csv")
    print("- edges_wagi_trudnosci.csv")


if __name__ == "__main__":
    main()

KeyError: 'averageParcelLoadKg'

In [5]:
import networkx as nx
import folium
import itertools
edgeColors = {
    ("ZG", "ZR"): "red",
    ("ZG", "ZA"): "blue",
    ("ZG", "SZP"): "green",
    ("ZG", "KOZ"): "purple",
    ("ZG", "NB"): "orange",
    ("ZG", "NS"): "black",

    ("NB", "SZP"): "pink",
    ("NB", "ZR"): "cadetblue",
    ("NB", "ZA"): "darkred",
    ("NB", "KOZ"): "darkgreen",
    ("NB", "NS"): "darkblue",

    ("ZR", "ZA"): "gray",
    ("ZR", "SZP"): "beige",
    ("ZR", "KOZ"): "lightblue",
    ("ZR", "NS"): "lightgreen",

    ("ZA", "SZP"): "darkpurple",
    ("ZA", "KOZ"): "darkbrown",
    ("ZA", "NS"): "lightred",

    ("SZP", "KOZ"): "darkcyan",
    ("SZP", "NS"): "olive",

    ("KOZ", "NS"): "navy",

    ("ZG", "ZG"): "brown"
}
#WIRRZCHLOKI

points = [
    {
        "id": "ZG",
        "name": "Zielona Góra",
        "lat": 51.9354800,
        "lon": 15.5064300
    },
    {
        "id": "NB",
        "name": "Nowogród Bobrzański",
        "lat": 51.798346,
        "lon": 15.236140
    },
    {
        "id": "ZR",
        "name": "Żary",
        "lat": 51.636,
        "lon": 15.138
    },
    {
        "id": "ZA",
        "name": "Żagań",
        "lat": 51.616591,
        "lon": 15.317660
    },
    {
        "id": "SZP",
        "name": "Szprotawa",
        "lat": 51.567,
        "lon": 15.538
    },
    {
        "id": "KOZ",
        "name": "Kożuchów",
        "lat": 51.745,
        "lon": 15.595
    },
    {
        "id": "NS",
        "name": "Nowa Sól",
        "lat": 51.8011,
        "lon": 15.7075
    }
]


G = nx.MultiGraph()


for point in points:
    G.add_node(
        point["id"],
        name=point["name"],
        lat=point["lat"],
        lon=point["lon"]
    )


# krawedzie 
# krawedzie 
# zg
G.add_edge("ZG", "ZR", czas=43, km=44.9, droga="DK27",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=4)

G.add_edge("ZG", "ZR", czas=49, km=48.7, droga="DK32 + DK27",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=4)


G.add_edge("ZG", "ZA", czas=46, km=46.2, droga="DK27 + DW295",
           typDrogi="DK + DW", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=3)

G.add_edge("ZG", "ZA", czas=56, km=58.2, droga="S3 + DW296",
           typDrogi="S + DW", predkoscMax=120, natezenieRuchu=4, jakoscDrogi=4)


G.add_edge("ZG", "SZP", czas=52, km=53.6, droga="S3 + DW297",
           typDrogi="S + DW", predkoscMax=120, natezenieRuchu=4, jakoscDrogi=4)

G.add_edge("ZG", "SZP", czas=54, km=58.1, droga="S3",
           typDrogi="S", predkoscMax=120, natezenieRuchu=4, jakoscDrogi=5)

G.add_edge("ZG", "SZP", czas=51, km=51.6, droga="DW283 + DW297",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


G.add_edge("ZG", "KOZ", czas=30, km=32, droga="S3 + DW297",
           typDrogi="S + DW", predkoscMax=120, natezenieRuchu=4, jakoscDrogi=4)

G.add_edge("ZG", "KOZ", czas=31, km=36.1, droga="S3",
           typDrogi="S", predkoscMax=120, natezenieRuchu=4, jakoscDrogi=5)

G.add_edge("ZG", "KOZ", czas=35, km=28.2, droga="DW283",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


G.add_edge("ZG", "NB", czas=25, km=25.5, droga="DW282 + DK27",
           typDrogi="DW + DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=3)


G.add_edge("ZG", "NS", czas=22, km=26.6, droga="S3",
           typDrogi="S", predkoscMax=120, natezenieRuchu=5, jakoscDrogi=5)

G.add_edge("ZG", "NS", czas=22, km=23.2, droga="S3 + Zielonogórska",
           typDrogi="S + miejska", predkoscMax=120, natezenieRuchu=5, jakoscDrogi=4)


# nowogród bobrzański
G.add_edge("NB", "SZP", czas=41, km=40.1, droga="DW295 + DK12",
           typDrogi="DW + DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=3)

G.add_edge("NB", "SZP", czas=50, km=50.2, droga="DK27 + DK12",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=4)

G.add_edge("NB", "SZP", czas=53, km=53.4, droga="DW290 + DW297",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


G.add_edge("NB", "ZR", czas=20, km=19.9, droga="DK27",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=4)

G.add_edge("NB", "ZR", czas=23, km=21.6, droga="DK27 + Obwodnica",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=4)


G.add_edge("NB", "ZA", czas=24, km=23.1, droga="DW295",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)

G.add_edge("NB", "ZA", czas=31, km=33.1, droga="DK27 + DK12",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=4)


G.add_edge("NB", "KOZ", czas=32, km=31.4, droga="DW290",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


G.add_edge("NB", "NS", czas=44, km=39.9, droga="DW290",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)

G.add_edge("NB", "NS", czas=41, km=54.4, droga="DK27 + S3",
           typDrogi="DK + S", predkoscMax=120, natezenieRuchu=4, jakoscDrogi=4)


# żary
G.add_edge("ZR", "ZA", czas=16, km=15.1, droga="DK12",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=4, jakoscDrogi=4)

G.add_edge("ZR", "ZA", czas=20, km=18.3, droga="Obwodnica + DK12",
           typDrogi="DK + miejska", predkoscMax=90, natezenieRuchu=4, jakoscDrogi=4)


G.add_edge("ZR", "SZP", czas=32, km=32.2, droga="DK12",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=4)


G.add_edge("ZR", "KOZ", czas=40, km=40.7, droga="DK12 + DW296",
           typDrogi="DK + DW", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=3)


G.add_edge("ZR", "NS", czas=55, km=52.4, droga="DW296",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


# żagań
G.add_edge("ZA", "SZP", czas=17, km=17.6, droga="DK12",
           typDrogi="DK", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=4)


G.add_edge("ZA", "KOZ", czas=25, km=26.1, droga="DW296",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


G.add_edge("ZA", "NS", czas=39, km=37.9, droga="DW296",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


# szprotawa
G.add_edge("SZP", "KOZ", czas=22, km=22.6, droga="DW297",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


G.add_edge("SZP", "NS", czas=36, km=34.3, droga="DW297",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=2, jakoscDrogi=3)


# kożuchów
G.add_edge("KOZ", "NS", czas=14, km=11.7, droga="DW297",
           typDrogi="DW", predkoscMax=90, natezenieRuchu=3, jakoscDrogi=3)


# self-loop
G.add_edge("ZG", "ZG", czas=2, km=3, droga="Lokalna",
           typDrogi="miejska", predkoscMax=50, natezenieRuchu=3, jakoscDrogi=3)


centerLat = sum(point["lat"] for point in points) / len(points)
centerLon = sum(point["lon"] for point in points) / len(points)


m = folium.Map(
    location=[centerLat, centerLon],
    zoom_start=9,
    tiles="CartoDB positron"
)


for nodeId, data in G.nodes(data=True):

    folium.Marker(
        location=[data["lat"], data["lon"]],
        tooltip=f"{nodeId} - {data['name']}"
    ).add_to(m)


for nodeA, nodeB, key, data in G.edges(keys=True, data=True):

    pointA = G.nodes[nodeA]
    pointB = G.nodes[nodeB]

    lat1 = pointA["lat"]
    lon1 = pointA["lon"]

    lat2 = pointB["lat"]
    lon2 = pointB["lon"]

    offset = (key - 1) * 0.03

    midLat = ((lat1 + lat2) / 2) + offset
    midLon = ((lon1 + lon2) / 2) - offset

    edgeKey = tuple(sorted([nodeA, nodeB]))
    color = edgeColors.get(edgeKey, "black")

    folium.PolyLine(
        locations=[
            [lat1, lon1],
            [midLat, midLon],
            [lat2, lon2]
        ],
        color=color,
        weight=4,
        tooltip=(
    f"{nodeA} ↔ {nodeB}<br>"
    f"droga = {data['droga']}<br>"
    f"typ drogi = {data['typDrogi']}<br>"
    f"predkosc max = {data['predkoscMax']} km/h<br>"
    f"natężenie ruchu = {data['natezenieRuchu']}/5<br>"
    f"jakość drogi = {data['jakoscDrogi']}/5<br>"
    f"km = {data['km']}<br>"
    f"czas = {data['czas']} min"
)
    ).add_to(m)


m.save("graf_wazony.html")


print("Mapa zapisana jako graf_wazony.html")
print()

print("KRAWĘDZIE:")
print()

for nodeA, nodeB, key, data in G.edges(keys=True, data=True):

    print(
        f"{nodeA} ↔ {nodeB} | "
        f"{data['droga']} | "
        f"typ: {data['typDrogi']} | "
        f"max: {data['predkoscMax']} km/h | "
        f"ruch: {data['natezenieRuchu']}/5 | "
        f"jakość: {data['jakoscDrogi']}/5 | "
        f"{data['km']} km | "
        f"{data['czas']} min"
    )


Mapa zapisana jako graf_wazony.html

KRAWĘDZIE:

ZG ↔ ZR | DK27 | typ: DK | max: 90 km/h | ruch: 3/5 | jakość: 4/5 | 44.9 km | 43 min
ZG ↔ ZR | DK32 + DK27 | typ: DK | max: 90 km/h | ruch: 3/5 | jakość: 4/5 | 48.7 km | 49 min
ZG ↔ ZA | DK27 + DW295 | typ: DK + DW | max: 90 km/h | ruch: 3/5 | jakość: 3/5 | 46.2 km | 46 min
ZG ↔ ZA | S3 + DW296 | typ: S + DW | max: 120 km/h | ruch: 4/5 | jakość: 4/5 | 58.2 km | 56 min
ZG ↔ SZP | S3 + DW297 | typ: S + DW | max: 120 km/h | ruch: 4/5 | jakość: 4/5 | 53.6 km | 52 min
ZG ↔ SZP | S3 | typ: S | max: 120 km/h | ruch: 4/5 | jakość: 5/5 | 58.1 km | 54 min
ZG ↔ SZP | DW283 + DW297 | typ: DW | max: 90 km/h | ruch: 2/5 | jakość: 3/5 | 51.6 km | 51 min
ZG ↔ KOZ | S3 + DW297 | typ: S + DW | max: 120 km/h | ruch: 4/5 | jakość: 4/5 | 32 km | 30 min
ZG ↔ KOZ | S3 | typ: S | max: 120 km/h | ruch: 4/5 | jakość: 5/5 | 36.1 km | 31 min
ZG ↔ KOZ | DW283 | typ: DW | max: 90 km/h | ruch: 2/5 | jakość: 3/5 | 28.2 km | 35 min
ZG ↔ NB | DW282 + DK27 | typ: DW + DK 

In [1]:
import pandas as pd

In [8]:
df = pd.read_csv("paczki_tydzien_04.05-08.05.2026.csv")

In [10]:
df['miasto_docelowe'].value_counts()

miasto_docelowe
Zielona Góra           17648
Nowa Sól                4656
Żary                    4339
Żagań                   3148
Szprotawa               1352
Nowogród Bobrzański      649
Kożuchów                 494
Name: count, dtype: int64

,id_paczki,data_dostawy,gabaryt,wysokosc_cm,szerokosc_cm,dlugosc_cm,waga_kg,miasto_docelowe,id_paczkomatu
0,ZG-050400000,2026-05-04,C,30,2,11,8,Zielona Góra,ZG-10
1,ZAR-050400001,2026-05-04,A,8,24,22,2,Żary,ZAR-13
2,ZG-050400002,2026-05-04,A,5,22,57,1,Zielona Góra,ZG-03
3,ZAR-050400003,2026-05-04,B,16,15,31,2,Żary,ZAR-03
4,ZG-050400004,2026-05-04,B,9,17,13,3,Zielona Góra,ZG-13
...,...,...,...,...,...,...,...,...,...
32281,NS-050805441,2026-05-08,C,22,22,9,7,Nowa Sól,NS-06
32282,ZG-050805442,2026-05-08,A,7,31,25,2,Zielona Góra,ZG-01
32283,NS-050805443,2026-05-08,C,29,21,14,15,Nowa Sól,NS-08
32284,ZG-050805444,2026-05-08,B,11,12,44,6,Zielona Góra,ZG-23
